# Paper 2 · Langkah 7: kondisi simetris

Di kondisi **riwayat_disamakan**, hanya karya manusia yang dikompresi ulang. Kuturin dkk. (2026) memperingatkan bahwa perbaikan satu sisi seperti itu bisa memindahkan jejak kompresi, bukan menghapusnya. Notebook ini menguji kekhawatiran itu.

**Kondisi simetris**: semua gambar (karya manusia, Latent Diffusion, Stable Diffusion) melewati urutan yang sama, yaitu diubah ke 256 × 256, disimpan JPEG kualitas 75 (4:2:0), lalu disimpan lagi JPEG kualitas 90 (4:4:4).

Yang dikerjakan:
1. **Matriks kompresi**: model dari tiga kondisi diuji pada gambar dari tiga versi kompresi. Kalau model membaca jejak kompresi, kinerjanya akan jatuh saat versi gambarnya ditukar.
2. Melatih detektor 5-fold pada kondisi simetris.
3. Peta occlusion (skala logit), uji deletion, dan elemen desain untuk kondisi simetris, dibandingkan dengan hasil sebelumnya.
4. Uji tanpa warna dan baseline ukuran desain pada kondisi simetris.

**Sebelum Run all:** menu **Runtime → Change runtime type → T4 GPU → Save**.

Hasil tiap bagian langsung disimpan ke Drive. Kalau sesi terputus, klik Run all lagi; bagian yang sudah selesai dilewati.

## 1. Persiapan dan cek GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, io, json, time, random, shutil, zipfile, platform
import numpy as np, pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from torchvision.transforms.functional import gaussian_blur
import cv2, scipy, sklearn
from scipy import stats, ndimage
from PIL import Image
import PIL

# Pengaturan (sama dengan langkah 3 sampai 6; jangan diubah tanpa mencatatnya di log kerja)
SEED = 42
INPUT = 224
PATCH, STRIDE = 32, 16
TOP = 0.20
N_RANDOM = 10
BLUR_KERNEL, BLUR_SIGMA = 21, 7.0
HUE_BINS, GLCM_LEVELS = 36, 32
CANNY_LOW, CANNY_HIGH = 100, 200
CHROMA_MIN = 0.10
MAX_EPOCHS, PATIENCE, BATCH, LR, LR_MIN, PRETRAINED = 30, 5, 32, 1e-4, 1e-6, True

PROJECT = Path('/content/drive/MyDrive/Paper2')
NOTES = PROJECT / 'catatan'
RESULTS = PROJECT / 'hasil'
RUNS = RESULTS / 'detektor_runs'
MODELS = RESULTS / 'models'
MAPS3 = RESULTS / 'peta_simetris'
ELEM3 = RESULTS / 'elemen_simetris'
CACHE = PROJECT / 'data_olahan'
LOCAL = Path('/content/work')
for p in [RUNS, MODELS, MAPS3, ELEM3, LOCAL]:
    p.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu = torch.cuda.get_device_name(0) if device == 'cuda' else 'TIDAK ADA'
print('GPU:', gpu)
assert device == 'cuda', 'GPU belum aktif. Buka Runtime → Change runtime type → pilih T4 GPU → Save, lalu Run all lagi.'
USE_AMP = device == 'cuda'

env = (f'tanggal: {time.strftime("%Y-%m-%d %H:%M")}\npython: {platform.python_version()}\ntorch: {torch.__version__}\n'
       f'torchvision: {torchvision.__version__}\nnumpy: {np.__version__}\nscipy: {scipy.__version__}\nsklearn: {sklearn.__version__}\n'
       f'opencv: {cv2.__version__}\npillow: {PIL.__version__}\ngpu: {gpu}\n'
       f'kondisi simetris: semua gambar -> 256x256 -> JPEG q75 4:2:0 -> JPEG q90 4:4:4\n')
(NOTES / 'lingkungan_langkah7.txt').write_text(env)
print(env)

## 2. Data: tiga versi kompresi dari gambar yang sama

- **normal**: disimpan sekali, JPEG kualitas 90.
- **riwayat_disamakan**: karya manusia dikompresi kualitas 75 dulu; gambar AI tidak.
- **simetris**: semua gambar dikompresi kualitas 75 dulu.

In [ ]:
manifest = pd.read_csv(NOTES / 'manifest_ai_artbench_sample_v2.csv')
assert len(manifest) == 3000 and 'fold' in manifest.columns, 'Jalankan notebook langkah 2 dulu.'
manifest['y'] = (manifest['label'] == 'ai').astype(int)
assert (RESULTS / 'rq2_ketahanan_elemen.csv').exists(), 'Jalankan notebook langkah 6 dulu.'

local_raw = LOCAL / 'raw'
raw_zip = CACHE / 'ai_artbench_sample_raw.zip'
t0 = time.time()
if not local_raw.exists():
    shutil.copy(raw_zip, LOCAL / 'raw.zip')
    with zipfile.ZipFile(LOCAL / 'raw.zip') as z:
        z.extractall(local_raw)
manifest['raw_path'] = manifest.apply(
    lambda r: str(local_raw / r['source'] / r['style'] / Path(r['file_drive']).name), axis=1)

def enc(im, q, sub):
    buf = io.BytesIO(); im.save(buf, 'JPEG', quality=q, subsampling=sub, optimize=False); return buf.getvalue()
def dec(b):
    im = Image.open(io.BytesIO(b)); im.load(); return im.convert('RGB')
def square256(im):
    im = im.convert('RGB'); w, h = im.size
    if w != h:
        s = min(w, h); left, top = (w - s) // 2, (h - s) // 2
        im = im.crop((left, top, left + s, top + s))
    if im.size != (256, 256):
        im = im.resize((256, 256), Image.BICUBIC)
    return im

def make_version(row, version):
    with Image.open(row.raw_path) as im:
        im.load(); im = im.convert('RGB')
    if version == 'riwayat_disamakan' and row.source == 'human':
        im = dec(enc(im, 75, 2))
    im = square256(im)
    if version == 'simetris':
        im = dec(enc(im, 75, 2))
    b = enc(im, 90, 0)
    out = dec(b)
    if out.size != (INPUT, INPUT):
        out = out.resize((INPUT, INPUT), Image.BICUBIC)
    return np.asarray(out, dtype=np.uint8), len(b)

VERSIONS = ['normal', 'riwayat_disamakan', 'simetris']
XH, XT, SIZE = {}, {}, {}
for v in VERSIONS:
    with ThreadPoolExecutor(8) as ex:
        res = list(ex.map(lambda r: make_version(r, v), manifest.itertuples()))
    XH[v] = np.stack([a for a, _ in res]); SIZE[v] = np.array([s for _, s in res]) / 1024
    XT[v] = torch.from_numpy(XH[v]).permute(0, 3, 1, 2).contiguous()
idx_of = {s: i for i, s in enumerate(manifest['sample_id'])}
src_all, folds, y_all = manifest['source'].values, manifest['fold'].values, manifest['y'].values
strata_all = (manifest['source'] + '|' + manifest['style']).values
print(f'Data siap ({time.time() - t0:.0f} detik):', {v: XH[v].shape for v in VERSIONS})

## 3. Fungsi model, latihan, dan peta

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
NPIX = INPUT * INPUT
FRACS = torch.arange(1, 11, device=device) * 0.05

def norm_t(xb):
    return ((xb.to(device).float().div(255) - MEAN) / STD).contiguous(memory_format=torch.channels_last)

def build_model(pretrained=False):
    w = torchvision.models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
    m = torchvision.models.resnet50(weights=w)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(device).to(memory_format=torch.channels_last)

def load_model(name):
    m = build_model(False)
    m.load_state_dict(torch.load(MODELS / f'{name}.pt', map_location=device))
    return m.eval()

@torch.no_grad()
def probs(model, xn, bs=256):
    out = []
    for i in range(0, len(xn), bs):
        with torch.autocast(device_type=device, enabled=USE_AMP):
            logits = model(xn[i:i + bs])
        out.append(torch.softmax(logits.float(), 1))
    return torch.cat(out)

@torch.no_grad()
def predict_ai(model, X, idx):
    return torch.cat([probs(model, norm_t(X[idx[i:i + 128]])) for i in range(0, len(idx), 128)]).cpu().numpy()

@torch.no_grad()
def logits32(model, xn, bs=256):
    return torch.cat([model(xn[i:i + bs]).float() for i in range(0, len(xn), bs)])

# --- latihan (sama dengan langkah 3 dan 4) ---
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def prep_train(xb):
    xb = xb.to(device, non_blocking=True).float().div_(255)
    flip = torch.rand(xb.size(0), device=device) < 0.5
    xb[flip] = xb[flip].flip(-1)
    return ((xb - MEAN) / STD).contiguous(memory_format=torch.channels_last)

@torch.no_grad()
def val_loss(model, X, idx):
    model.eval(); tot = 0.0
    for i in range(0, len(idx), 128):
        b = idx[i:i + 128]
        with torch.autocast(device_type=device, enabled=USE_AMP):
            logits = model(norm_t(X[b]))
        tot += F.cross_entropy(logits.float(), torch.as_tensor(y_all[b], device=device), reduction='sum').item()
    return tot / len(idx)

def train_model(X, tr_idx, va_idx, seed):
    set_seed(seed)
    model = build_model(PRETRAINED)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS, eta_min=LR_MIN)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    rng = np.random.default_rng(seed)
    best, best_state, bad, hist = float('inf'), None, 0, []
    for ep in range(MAX_EPOCHS):
        model.train(); perm = rng.permutation(tr_idx)
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device, enabled=USE_AMP):
                loss = F.cross_entropy(model(prep_train(X[b])), torch.as_tensor(y_all[b], device=device))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        v = val_loss(model, X, va_idx); hist.append(round(v, 4))
        if v < best - 1e-4:
            best, bad = v, 0
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model.eval(), hist

def inner_split(tr_idx, seed):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=seed)
    try:
        a, b = next(sss.split(tr_idx, strata_all[tr_idx]))
    except ValueError:
        a, b = next(sss.split(tr_idx, src_all[tr_idx]))
    return tr_idx[a], tr_idx[b]

def source_rates(p_ai, idx):
    pred = (p_ai >= 0.5).astype(int); s = src_all[idx]; y = y_all[idx]; out = {'akurasi': float((pred == y).mean())}
    if len(np.unique(y)) == 2: out['auc'] = float(roc_auc_score(y, p_ai))
    for k, name in (('human', 'benar_manusia'), ('latent_diffusion', 'benar_LD'), ('stable_diffusion', 'benar_SD')):
        if (s == k).any(): out[name] = float((pred[s == k] == y[s == k]).mean())
    return out

# --- peta occlusion skala logit dan uji deletion (sama dengan langkah 6) ---
pos = list(range(0, INPUT - PATCH + 1, STRIDE)); N1 = len(pos)
M1 = torch.zeros(N1 * N1, 1, INPUT, INPUT, dtype=torch.bool, device=device); k = 0
for i in pos:
    for j in pos:
        M1[k, 0, i:i + PATCH, j:j + PATCH] = True; k += 1

@torch.no_grad()
def occlusion_logit(model, xn, pred):
    out, m0s = [], []
    for i in range(len(xn)):
        x = xn[i:i + 1]; c = int(pred[i]); mu = x.mean((2, 3), keepdim=True)
        z0 = logits32(model, x)[0]; m0 = z0[c] - z0[1 - c]
        z = logits32(model, torch.where(M1, mu, x))
        out.append((m0 - (z[:, c] - z[:, 1 - c])).reshape(N1, N1)); m0s.append(m0)
    return torch.stack(out), torch.stack(m0s)

def patch_to_cells(d):
    kk, n, C = PATCH // STRIDE, d.shape[-1], INPUT // STRIDE
    acc = torch.zeros(*d.shape[:-2], C, C, device=d.device); cnt = torch.zeros(C, C, device=d.device)
    for a in range(kk):
        for b in range(kk):
            acc[..., a:a + n, b:b + n] += d; cnt[a:a + n, b:b + n] += 1
    return acc / cnt.clamp(min=1)

def up(m):
    return F.interpolate(m[:, None].float(), size=(INPUT, INPUT), mode='bilinear', align_corners=False)[:, 0]

def rank_desc(maps, gen):
    flat = maps.reshape(len(maps), -1)
    flat = flat + 1e-7 * flat.abs().amax(1, keepdim=True).clamp(min=1e-12) * torch.rand(flat.shape, device=flat.device, generator=gen)
    order = flat.argsort(1, descending=True)
    rank = torch.empty_like(order)
    rank.scatter_(1, order, torch.arange(order.shape[1], device=flat.device).expand_as(order))
    return rank

yy, xx = torch.meshgrid(torch.arange(INPUT, device=device), torch.arange(INPUT, device=device), indexing='ij')
c0 = (INPUT - 1) / 2
CENTRE = 1 - torch.sqrt((yy - c0) ** 2 + (xx - c0) ** 2) / (c0 * 2 ** 0.5)

@torch.no_grad()
def deletion(model, x, maps, c, p0, gen):
    rank = rank_desc(maps, gen)
    thr = (FRACS * NPIX).long()
    mask = (rank[:, None, :] < thr[None, :, None]).reshape(-1, 1, INPUT, INPUT)
    out = {}
    for name, rep in (('blur', gaussian_blur(x, [BLUR_KERNEL, BLUR_KERNEL], [BLUR_SIGMA, BLUR_SIGMA])), ('mean', x.mean((2, 3), keepdim=True))):
        p = probs(model, torch.where(mask, rep, x))[:, c].reshape(len(maps), -1)
        out[name] = (0.5 * p0 + p[:, :-1].sum(1) + 0.5 * p[:, -1]) / len(FRACS)
    return out
print('Fungsi siap. Posisi kotak occlusion:', N1, 'x', N1)

## 4. Latih detektor pada kondisi simetris

Lima model 5-fold (bobot disimpan) dan dua model lintas generator (hanya angka kinerjanya).

In [ ]:
SHORT = {'latent_diffusion': 'LD', 'stable_diffusion': 'SD'}
X = XT['simetris']
for k in range(5):
    name = f'simetris_fold{k}'
    if (MODELS / f'{name}.pt').exists() and (RUNS / f'{name}.json').exists():
        print(f'{name}: sudah selesai, dilewati.'); continue
    t0 = time.time(); tr_idx, te_idx = np.where(folds != k)[0], np.where(folds == k)[0]
    tr, va = inner_split(tr_idx, SEED + k)
    model, hist = train_model(X, tr, va, SEED + k)
    p = predict_ai(model, X, te_idx)[:, 1]
    rec = {'nama': name, 'epoch': len(hist), 'val_loss': hist, 'detik': round(time.time() - t0), **source_rates(p, te_idx),
           'prediksi': [{'sample_id': manifest['sample_id'].iat[i], 'p_ai': round(float(pi), 6)} for i, pi in zip(te_idx, p)]}
    torch.save(model.state_dict(), MODELS / f'{name}.pt'); (RUNS / f'{name}.json').write_text(json.dumps(rec))
    print(f"{name}: akurasi {rec['akurasi']:.3f}, AUC {rec['auc']:.3f}, {rec['epoch']} epoch, {rec['detik']} detik")
    del model; torch.cuda.empty_cache()

human_tr = np.where((src_all == 'human') & (folds != 4))[0]; human_te = np.where((src_all == 'human') & (folds == 4))[0]
for a, b in [('latent_diffusion', 'stable_diffusion'), ('stable_diffusion', 'latent_diffusion')]:
    name = f'lintas_simetris_latih{SHORT[a]}_uji{SHORT[b]}'
    if (RUNS / f'{name}.json').exists():
        print(f'{name}: sudah selesai, dilewati.'); continue
    t0 = time.time()
    tr_idx = np.concatenate([human_tr, np.where(src_all == a)[0]]); te_idx = np.concatenate([human_te, np.where(src_all == b)[0]])
    tr, va = inner_split(tr_idx, SEED + 10)
    model, hist = train_model(X, tr, va, SEED + 10)
    p = predict_ai(model, X, te_idx)[:, 1]
    rec = {'nama': name, 'epoch': len(hist), 'detik': round(time.time() - t0), **source_rates(p, te_idx)}
    (RUNS / f'{name}.json').write_text(json.dumps(rec))
    print(f"{name}: akurasi {rec['akurasi']:.3f}, AUC {rec['auc']:.3f}, {rec['epoch']} epoch, {rec['detik']} detik")
    del model; torch.cuda.empty_cache()

## 5. Matriks kompresi dan uji tanpa warna

**Matriks kompresi**: model dari tiap kondisi diuji pada tiga versi gambar. Baris = kondisi saat model dilatih; kolom = versi gambar saat diuji. Kalau angka jatuh di luar diagonal, model membaca jejak kompresi.

In [ ]:
mat_file = RESULTS / 'rq1_matriks_kompresi.csv'
if mat_file.exists():
    MAT = pd.read_csv(mat_file); print('Matriks kompresi: sudah selesai, dilewati.')
else:
    rows = []
    for fam in VERSIONS:
        for k in range(5):
            model = load_model(f'{fam}_fold{k}'); te = np.where(folds == k)[0]
            for v in VERSIONS:
                rows.append({'model': fam, 'gambar': v, 'fold': k, **source_rates(predict_ai(model, XT[v], te)[:, 1], te)})
            del model; torch.cuda.empty_cache()
    MAT = pd.DataFrame(rows); MAT.to_csv(mat_file, index=False)
M = MAT.groupby(['model', 'gambar'])[['akurasi', 'auc', 'benar_manusia', 'benar_LD', 'benar_SD']].mean().reindex(
    pd.MultiIndex.from_product([VERSIONS, VERSIONS], names=['model', 'gambar']))
print('\nX. MATRIKS KOMPRESI (rata-rata 5 fold). model = kondisi latihan; gambar = versi yang diuji')
print(M.round(3).to_string())

def transform(xb, kind):
    x = xb.to(device).float()
    gray = (0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]).expand(-1, 3, -1, -1)
    if kind == 'hitam-putih': x = gray
    elif kind == 'saturasi 50%': x = 0.5 * x + 0.5 * gray
    elif kind == 'buram ringan': x = gaussian_blur(x, [13, 13], [2.0, 2.0])
    elif kind == 'buram kuat': x = gaussian_blur(x, [25, 25], [4.0, 4.0])
    return ((x.clamp(0, 255) / 255 - MEAN) / STD).contiguous(memory_format=torch.channels_last)

KINDS = ['asli', 'hitam-putih', 'saturasi 50%', 'buram ringan', 'buram kuat']
abl_file = RESULTS / 'rq1_uji_warna_detail_simetris_per_fold.csv'
if abl_file.exists():
    A = pd.read_csv(abl_file); print('\nUji warna dan detail (simetris): sudah selesai, dilewati.')
else:
    rows = []
    for k in range(5):
        model = load_model(f'simetris_fold{k}'); te = np.where(folds == k)[0]
        for kind in KINDS:
            p = torch.cat([probs(model, transform(XT['simetris'][te[i:i + 128]], kind)) for i in range(0, len(te), 128)])[:, 1].cpu().numpy()
            rows.append({'fold': k, 'perubahan': kind, **source_rates(p, te)})
        del model; torch.cuda.empty_cache()
    A = pd.DataFrame(rows); A.to_csv(abl_file, index=False)
print('\nW2. UJI TANPA WARNA DAN TANPA DETAIL, model simetris (rata-rata 5 fold)')
print(A.groupby('perubahan')[['akurasi', 'auc', 'benar_manusia', 'benar_LD', 'benar_SD']].mean().reindex(KINDS).round(3).to_string())

## 6. Peta occlusion dan uji deletion untuk kondisi simetris (bagian terlama)

Lima bagian, satu per fold. Peta dihitung pada skala logit dengan presisi penuh, sama seperti langkah 6.

In [ ]:
def process_maps(k):
    name = f'simetris_fold{k}'; f = MAPS3 / f'{name}.npz'
    if f.exists():
        print(f'{name}: sudah selesai, dilewati.'); return
    t0 = time.time(); model = load_model(name); idx = np.where(folds == k)[0]; n = len(idx); X = XT['simetris']
    gen = torch.Generator(device=device); gen.manual_seed(SEED + sum(map(ord, name)))
    P, OCC, M0 = [], [], []
    dele = {f'{r}_{m}': [] for r in ('blur', 'mean') for m in ('occ', 'centre', 'random')}
    fm = max(1, INPUT // 32)
    for s in range(0, n, 16):
        b = idx[s:s + 16]; xn = norm_t(X[b]); pr = probs(model, xn); pred = pr.argmax(1)
        d, m0 = occlusion_logit(model, xn, pred.tolist())
        u = up(patch_to_cells(d).clamp(min=0))
        for i in range(len(b)):
            rnd = up(torch.rand(N_RANDOM, fm, fm, device=device, generator=gen))
            res = deletion(model, xn[i:i + 1], torch.cat([u[i:i + 1], CENTRE[None], rnd]), int(pred[i]), pr[i, pred[i]], gen)
            for r in ('blur', 'mean'):
                a = res[r].cpu()
                dele[f'{r}_occ'].append(a[0].item()); dele[f'{r}_centre'].append(a[1].item()); dele[f'{r}_random'].append(a[2:].mean().item())
        P.append(pr.cpu()); OCC.append(d.cpu()); M0.append(m0.cpu())
    P = torch.cat(P).numpy()
    np.savez_compressed(f, sample_id=manifest['sample_id'].values[idx].astype(str), p=P, pred=P.argmax(1),
                        margin0=torch.cat(M0).numpy().astype(np.float32), occ_logit=torch.cat(OCC).numpy().astype(np.float32),
                        **{'del_' + kk: np.array(v, dtype=np.float32) for kk, v in dele.items()})
    print(f'{name}: {n} gambar, {time.time() - t0:.0f} detik')
    del model; torch.cuda.empty_cache()

for k in range(5):
    process_maps(k)

## 7. Elemen desain pada kondisi simetris

In [ ]:
NPIX = INPUT * INPUT
MEASURES = ['hue_entropy', 'saturation', 'colourfulness', 'brightness', 'contrast', 'gradient',
            'edge_density', 'glcm_contrast', 'glcm_homogeneity', 'glcm_energy', 'glcm_correlation']
LABEL = {'hue_entropy': 'Hue: entropi', 'saturation': 'Warna: saturasi', 'colourfulness': 'Warna: kekayaan',
         'brightness': 'Nada: kecerahan', 'contrast': 'Nada: kontras', 'gradient': 'Nada: gradasi',
         'edge_density': 'Garis: kepadatan tepi', 'glcm_contrast': 'Tekstur: kontras',
         'glcm_homogeneity': 'Tekstur: homogenitas', 'glcm_energy': 'Tekstur: energi', 'glcm_correlation': 'Tekstur: korelasi'}

# Pasangan piksel bertetangga untuk GLCM (4 arah, jarak 1)
_flat = np.arange(NPIX).reshape(INPUT, INPUT)
PAIRS = []
for dy, dx in [(0, 1), (1, 0), (1, 1), (1, -1)]:
    a = _flat[max(0, -dy):INPUT - max(0, dy), max(0, -dx):INPUT - max(0, dx)]
    b = _flat[max(0, dy):INPUT - max(0, -dy), max(0, dx):INPUT - max(0, -dx)]
    PAIRS.append((a.ravel(), b.ravel()))
_lv = np.arange(GLCM_LEVELS, dtype=np.float64)
_I, _J = np.meshgrid(_lv, _lv, indexing='ij')
_D2 = (_I - _J) ** 2

def precompute(img):
    f = {}
    rgb = img.astype(np.float32)
    hsv = cv2.cvtColor(rgb / 255.0, cv2.COLOR_RGB2HSV)            # H 0-360, S 0-1, V 0-1
    f['hbin'] = np.minimum((hsv[..., 0] / 360.0 * HUE_BINS).astype(np.int64), HUE_BINS - 1).ravel()
    f['sat'] = hsv[..., 1].ravel(); f['val'] = hsv[..., 2].ravel()
    f['chroma_ok'] = (f['sat'] > CHROMA_MIN) & (f['val'] > CHROMA_MIN)
    f['rg'] = (rgb[..., 0] - rgb[..., 1]).ravel()
    f['yb'] = (0.5 * (rgb[..., 0] + rgb[..., 1]) - rgb[..., 2]).ravel()
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    f['lum'] = gray.astype(np.float32).ravel()
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    f['grad'] = np.sqrt(gx ** 2 + gy ** 2).ravel()
    f['edge'] = (cv2.Canny(gray, CANNY_LOW, CANNY_HIGH) > 0).ravel()
    q = (gray.astype(np.int64) * GLCM_LEVELS // 256).ravel()
    f['codes'] = [q[a] * GLCM_LEVELS + q[b] for a, b in PAIRS]
    return f

def region_measures(f, m):
    out = dict.fromkeys(MEASURES, np.nan)
    if m.sum() < 100:
        return out
    sel = m & f['chroma_ok']
    if sel.sum() >= 50:
        h = np.bincount(f['hbin'][sel], minlength=HUE_BINS).astype(np.float64); p = h / h.sum(); p = p[p > 0]
        out['hue_entropy'] = float(-(p * np.log2(p)).sum())
    out['saturation'] = float(f['sat'][m].mean())
    rg, yb = f['rg'][m], f['yb'][m]
    out['colourfulness'] = float(np.sqrt(rg.std() ** 2 + yb.std() ** 2) + 0.3 * np.sqrt(rg.mean() ** 2 + yb.mean() ** 2))
    out['brightness'] = float(f['val'][m].mean())
    out['contrast'] = float(f['lum'][m].std())
    out['gradient'] = float(f['grad'][m].mean())
    out['edge_density'] = float(f['edge'][m].mean())
    P = np.zeros(GLCM_LEVELS * GLCM_LEVELS)
    for (a, b), code in zip(PAIRS, f['codes']):
        ok = m[a] & m[b]
        P += np.bincount(code[ok], minlength=GLCM_LEVELS * GLCM_LEVELS)
    if P.sum() > 0:
        P = P.reshape(GLCM_LEVELS, GLCM_LEVELS); P = P + P.T; P /= P.sum()
        out['glcm_contrast'] = float((P * _D2).sum())
        out['glcm_homogeneity'] = float((P / (1 + _D2)).sum())
        out['glcm_energy'] = float(np.sqrt((P ** 2).sum()))
        mu = (P.sum(1) * _lv).sum(); var = (P.sum(1) * (_lv - mu) ** 2).sum()
        out['glcm_correlation'] = float((P * (_I - mu) * (_J - mu)).sum() / var) if var > 1e-12 else np.nan
    return out

GRID = max(1, INPUT // 32)
def composition(m):
    m2 = m.reshape(INPUT, INPUT); ys, xs = np.nonzero(m2); c = (INPUT - 1) / 2
    cell = m2[:GRID * (INPUT // GRID), :GRID * (INPUT // GRID)].reshape(GRID, INPUT // GRID, GRID, INPUT // GRID).sum((1, 3)).ravel().astype(np.float64)
    p = cell / cell.sum(); p = p[p > 0]
    lab, ncomp = ndimage.label(m2, structure=np.ones((3, 3)))
    sizes = np.bincount(lab.ravel())[1:]
    return {'comp_centroid_dist': float(np.sqrt((ys.mean() - c) ** 2 + (xs.mean() - c) ** 2) / (c * 2 ** 0.5)),
            'comp_spatial_entropy': float(-(p * np.log2(p)).sum() / np.log2(GRID * GRID)) if GRID > 1 else np.nan,
            'comp_n_components': int(ncomp), 'comp_largest_share': float(sizes.max() / sizes.sum())}

def patch_to_cells_cpu(d):
    k, n, C = PATCH // STRIDE, d.shape[-1], INPUT // STRIDE
    acc = torch.zeros(*d.shape[:-2], C, C); cnt = torch.zeros(C, C)
    for a in range(k):
        for b in range(k):
            acc[..., a:a + n, b:b + n] += d; cnt[a:a + n, b:b + n] += 1
    return acc / cnt.clamp(min=1)

def up(m):
    return F.interpolate(m[:, None].float(), size=(INPUT, INPUT), mode='bilinear', align_corners=False)[:, 0]

def rank_maps(maps, rng):
    n = len(maps); rank = np.empty((n, NPIX), dtype=np.uint16)
    for s in range(0, n, 200):
        flat = maps[s:s + 200].reshape(-1, NPIX).astype(np.float64)
        flat = flat + 1e-9 * rng.random(flat.shape)                 # pemecah nilai kembar
        order = np.argsort(-flat, axis=1)
        np.put_along_axis(rank[s:s + 200], order, np.arange(NPIX, dtype=np.uint16)[None].repeat(len(order), 0), axis=1)
    return rank
ALL = np.ones(NPIX, dtype=bool)

def process_elem(k):
    name = f'simetris_fold{k}'; out_file = ELEM3 / f'{name}.csv'
    if out_file.exists():
        print(f'{name}: sudah selesai, dilewati.'); return
    t0 = time.time()
    z = np.load(MAPS3 / f'{name}.npz', allow_pickle=True)
    ids = z['sample_id'].astype(str); idx = np.array([idx_of[s] for s in ids]); pred = z['pred'].astype(int); n = len(idx)
    rng = np.random.default_rng(SEED + sum(map(ord, name)))
    occ = torch.from_numpy(z['occ_logit']).float(); cells = patch_to_cells_cpu(occ).clamp(min=0)
    empty = (cells.amax((1, 2)) <= 0).numpy(); ranks = rank_maps(up(cells).numpy(), rng)
    partner = np.arange(n); s_here = src_all[idx]
    for s in np.unique(s_here):
        loc = rng.permutation(np.where(s_here == s)[0])
        if len(loc) > 1:
            partner[loc] = np.roll(loc, -1)
    rows = []
    for j in range(n):
        f = precompute(XH['simetris'][idx[j]])
        row = {'bagian': name, 'sample_id': ids[j], 'pred': int(pred[j]), 'peta_kosong': bool(empty[j]),
               'margin0': float(z['margin0'][j]), 'occ_max_drop_logit': float(occ[j].max())}
        row.update({kk: float(z[kk][j]) for kk in z.files if kk.startswith('del_')})
        row.update({f'whole_{a}': v for a, v in region_measures(f, ALL).items()})
        if not empty[j]:
            m = ranks[j] < TOP * NPIX
            for tag, mask in (('in', m), ('out', ~m)):
                row.update({f'{tag}_{a}': v for a, v in region_measures(f, mask).items()})
            if partner[j] != j and not empty[partner[j]]:
                ms = ranks[partner[j]] < TOP * NPIX
                for tag, mask in (('shin', ms), ('shout', ~ms)):
                    row.update({f'{tag}_{a}': v for a, v in region_measures(f, mask).items()})
        rows.append(row)
    pd.DataFrame(rows).to_csv(out_file, index=False)
    print(f'{name}: {n} gambar, {time.time() - t0:.0f} detik')

for k in range(5):
    process_elem(k)

## 8. Ringkasan

- **S1**: kinerja detektor pada kondisi simetris.
- **S2**: uji deletion.
- **S3**: elemen desain, tiga kondisi berdampingan: simetris | riwayat_disamakan | normal.
- **S4**: seberapa cocok hasil simetris dengan dua kondisi lain.
- **S5 dan S6**: ukuran seluruh gambar dan baseline ukuran desain pada kondisi simetris.

In [ ]:
SRC = {'human': 'manusia', 'latent_diffusion': 'LD', 'stable_diffusion': 'SD'}
runs = [json.loads((RUNS / f'simetris_fold{k}.json').read_text()) for k in range(5)]
R = pd.DataFrame([{kk: r[kk] for kk in ('akurasi', 'auc', 'benar_manusia', 'benar_LD', 'benar_SD', 'epoch')} for r in runs])
print('S1. DETEKTOR PADA KONDISI SIMETRIS (5-fold): rata-rata (simpangan baku)')
print((R.mean().map('{:.3f}'.format) + ' (' + R.std(ddof=0).map('{:.3f}'.format) + ')').to_string())
print('\nS1b. LINTAS GENERATOR, kondisi simetris')
print(pd.DataFrame([json.loads(f.read_text()) for f in sorted(RUNS.glob('lintas_simetris_*.json'))]).drop(columns=['detik']).round(3).to_string(index=False))

rows = []
for v in VERSIONS:
    row = {'versi gambar': v}
    for name, srcs in (('manusia vs LD', ['human', 'latent_diffusion']), ('manusia vs SD', ['human', 'stable_diffusion'])):
        sel = np.isin(src_all, srcs); aucs = []
        for k in range(5):
            tr, te = sel & (folds != k), sel & (folds == k)
            mdl = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(SIZE[v][tr].reshape(-1, 1), y_all[tr])
            aucs.append(roc_auc_score(y_all[te], mdl.predict_proba(SIZE[v][te].reshape(-1, 1))[:, 1]))
        row[name] = np.mean(aucs)
    for s in SRC: row[f'median KB {SRC[s]}'] = np.median(SIZE[v][src_all == s])
    rows.append(row)
print('\nS0. UKURAN FILE SAJA sebagai penebak kelas (AUC, regresi logistik) dan median ukuran file')
print(pd.DataFrame(rows).round(3).to_string(index=False))

df = pd.concat([pd.read_csv(f) for f in sorted(ELEM3.glob('*.csv'))], ignore_index=True)
df = df.join(manifest.set_index('sample_id')[['source', 'style', 'fold', 'y']], on='sample_id')
df['kelompok'] = '5-fold: ' + df.source.map(SRC) + np.where(df.pred == df.y, ' benar', ' salah')
df.to_csv(RESULTS / 'rq2_elemen_simetris_per_gambar.csv', index=False)
GROUPS = ['5-fold: manusia benar', '5-fold: LD benar', '5-fold: SD benar']
print('\nBagian selesai:', df.bagian.nunique(), 'dari 5 | baris:', len(df))
print(df.groupby('kelompok').agg(n=('sample_id', 'size'), peta_kosong=('peta_kosong', 'sum'), median_selisih_logit=('margin0', 'median'),
                                 median_turun_terbesar=('occ_max_drop_logit', 'median')).round(3).to_string())

d2 = df[df.kelompok.isin(GROUPS)].groupby('kelompok')[['del_blur_occ', 'del_blur_centre', 'del_blur_random', 'del_mean_occ', 'del_mean_centre', 'del_mean_random']].mean().reindex(GROUPS)
d2['lulus'] = np.where((d2.del_blur_occ < d2.del_blur_centre) & (d2.del_blur_occ < d2.del_blur_random) &
                       (d2.del_mean_occ < d2.del_mean_centre) & (d2.del_mean_occ < d2.del_mean_random), 'ya', 'tidak')
d2.columns = ['blur:occ', 'blur:tengah', 'blur:acak', 'rata:occ', 'rata:tengah', 'rata:acak', 'lulus']
d2.to_csv(RESULTS / 'rq2_simetris_deletion.csv')
print('\nS2. UJI DELETION, kondisi simetris (makin kecil makin setia)')
print(d2.round(3).to_string())

def paired_effect(x):
    x = np.asarray(x, dtype=float); x = x[~np.isnan(x)]; x = x[x != 0]
    if len(x) < 10:
        return np.nan, np.nan, len(x)
    r = stats.rankdata(np.abs(x)); pos_, neg_ = r[x > 0].sum(), r[x < 0].sum()
    return (pos_ - neg_) / (pos_ + neg_), stats.wilcoxon(x).pvalue, len(x)
def bh(p):
    p = np.asarray(p, dtype=float); out = np.full(len(p), np.nan); ok = ~np.isnan(p)
    if ok.sum() == 0: return out
    pv = p[ok]; order = np.argsort(pv); ranked = pv[order] * len(pv) / (np.arange(len(pv)) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]; res = np.empty(len(pv)); res[order] = np.minimum(ranked, 1)
    out[ok] = res; return out

rows = []
for grp, g in df[~df.peta_kosong & df.kelompok.isin(GROUPS)].groupby('kelompok'):
    for m in MEASURES:
        r, p, n = paired_effect((g[f'in_{m}'] - g[f'out_{m}']) - (g[f'shin_{m}'] - g[f'shout_{m}']))
        rows.append({'kelompok': grp, 'ukuran': m, 'n': n, 'r_sim': r, 'p': p, 'median_dalam': g[f'in_{m}'].median(), 'median_luar': g[f'out_{m}'].median()})
S = pd.DataFrame(rows); S['sig_sim'] = bh(S.p.values) < 0.05
H = pd.read_csv(RESULTS / 'rq2_ketahanan_elemen.csv')[['kelompok', 'ukuran', 'r_logit', 'bermakna_logit']].rename(columns={'r_logit': 'r_riw', 'bermakna_logit': 'sig_riw'})
Nn = pd.read_csv(RESULTS / 'rq2_elemen_ringkasan.csv')
Nn = Nn[(Nn.keluarga == 'normal') & (Nn.metode == 'occlusion') & np.isclose(Nn.ambang, 0.2)][['kelompok', 'ukuran', 'r', 'bermakna']]
Nn['kelompok'] = Nn.kelompok.str.replace('normal:', '5-fold:', regex=False); Nn = Nn.rename(columns={'r': 'r_nor', 'bermakna': 'sig_nor'})
C = S.merge(H, on=['kelompok', 'ukuran'], how='left').merge(Nn, on=['kelompok', 'ukuran'], how='left')
C.to_csv(RESULTS / 'rq2_simetris_elemen.csv', index=False)

def cell(r, sig):
    return ('nan' if pd.isna(r) else f'{r:+.2f}') + ('*' if (sig is True or sig == 1) else ' ')
C['sel'] = [cell(a, b) + '|' + cell(c, d) + '|' + cell(e, f) for a, b, c, d, e, f in zip(C.r_sim, C.sig_sim, C.r_riw, C.sig_riw, C.r_nor, C.sig_nor)]
pv = C.pivot(index='ukuran', columns='kelompok', values='sel').reindex(MEASURES).reindex(columns=GROUPS)
pv.index = [LABEL[i] for i in pv.index]; pv.columns = ['manusia benar', 'LD benar', 'SD benar']
print('\nS3. ELEMEN DESAIN (occlusion, 20%): r simetris | r riwayat_disamakan | r normal   (* = bermakna)')
print(pv.to_string())

rows = []
for grp, g in C.groupby('kelompok'):
    for other, rc, sc in (('riwayat_disamakan', 'r_riw', 'sig_riw'), ('normal', 'r_nor', 'sig_nor')):
        sig = g[g.sig_sim | (g[sc] == True)]
        rows.append({'kelompok': grp, 'dibanding': other, 'korelasi_r': np.corrcoef(g.r_sim, g[rc])[0, 1], 'selisih_r_terbesar': (g.r_sim - g[rc]).abs().max(),
                     'ukuran_bermakna': len(sig), 'arah_sama': int((np.sign(sig.r_sim) == np.sign(sig[rc])).sum())})
print('\nS4. KECOCOKAN hasil simetris dengan kondisi lain')
print(pd.DataFrame(rows).set_index(['kelompok', 'dibanding']).reindex(GROUPS, level=0).round(3).to_string())

t = S.pivot(index='ukuran', columns='kelompok', values='median_dalam').reindex(MEASURES).reindex(columns=GROUPS).round(3).astype(str) + ' | ' + \
    S.pivot(index='ukuran', columns='kelompok', values='median_luar').reindex(MEASURES).reindex(columns=GROUPS).round(3).astype(str)
t.index = [LABEL[i] for i in t.index]; t.columns = ['manusia benar', 'LD benar', 'SD benar']
print('\nS3b. BESARAN pada kondisi simetris: median dalam | median luar')
print(t.to_string())

w = df.drop_duplicates('sample_id').reset_index(drop=True)
wr = []
for m in MEASURES:
    row = {'ukuran': LABEL[m]}; hv = w[w.source == 'human'][f'whole_{m}'].dropna(); row['median manusia'] = hv.median()
    for s in ('latent_diffusion', 'stable_diffusion'):
        av = w[w.source == s][f'whole_{m}'].dropna(); u, p = stats.mannwhitneyu(av, hv, alternative='two-sided')
        row[f'median {SRC[s]}'] = av.median(); row[f'delta {SRC[s]}'] = 2 * u / (len(av) * len(hv)) - 1
    wr.append(row)
print('\nS5. SELURUH GAMBAR pada kondisi simetris (delta positif = AI lebih tinggi daripada manusia)')
print(pd.DataFrame(wr).set_index('ukuran').round(3).to_string())

SETS = {'semua (11)': MEASURES, 'warna (3)': ['hue_entropy', 'saturation', 'colourfulness'], 'nada (3)': ['brightness', 'contrast', 'gradient'],
        'garis dan tekstur (5)': ['edge_density', 'glcm_contrast', 'glcm_homogeneity', 'glcm_energy', 'glcm_correlation']}
def fit_pred(tr, te, feats):
    cols = [f'whole_{m}' for m in feats]; mdl = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
    mdl.fit(tr[cols].fillna(tr[cols].median()), tr.y); return mdl.predict_proba(te[cols].fillna(tr[cols].median()))[:, 1]
rows = []
for sname, feats in SETS.items():
    row = {'ukuran': sname}
    for tname, srcs in (('AI vs manusia', list(SRC)), ('manusia vs LD', ['human', 'latent_diffusion']), ('manusia vs SD', ['human', 'stable_diffusion'])):
        d = w[w.source.isin(srcs)]
        row[tname] = np.mean([roc_auc_score(d[d.fold == k].y, fit_pred(d[d.fold != k], d[d.fold == k], feats)) for k in range(5)])
    for a, b in (('latent_diffusion', 'stable_diffusion'), ('stable_diffusion', 'latent_diffusion')):
        tr = w[((w.source == 'human') & (w.fold != 4)) | (w.source == a)]; te = w[((w.source == 'human') & (w.fold == 4)) | (w.source == b)]
        p = fit_pred(tr, te, feats); row[f'latih {SRC[a]} uji {SRC[b]}: AI dikenali'] = (p[te.y.values == 1] >= 0.5).mean()
    rows.append(row)
D = pd.DataFrame(rows).set_index('ukuran'); D.to_csv(RESULTS / 'rq2_simetris_baseline_ukuran_desain.csv')
print('\nS6. BASELINE UKURAN DESAIN pada kondisi simetris (AUC 5-fold; dua kolom terakhir = uji lintas generator)')
print(D.round(3).to_string())
print('\nTersimpan di', RESULTS)

## Selesai

Kalau tertulis "Bagian selesai: 5 dari 5" dan tabel S0 sampai S6 muncul, semua beres. Tekan **Cmd+S**, lalu kabari Claude.